In [ ]:
import json, os, sys, glob, subprocess, torch
from pathlib import Path

# cell 0: hardware smoke, fail fast
names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("torch",torch.__version__,"cuda",torch.version.cuda)
print("device_count",torch.cuda.device_count(),names)
assert torch.cuda.device_count()==2, "Expected 2x T4 (GPU T4 x2)"
for i in range(2):
    with torch.cuda.device(i):
        x=torch.randn(4096,4096,device=f"cuda:{i}",dtype=torch.float16)
        s=(x@x).float().sum().item(); torch.cuda.synchronize(i)
    print(f"cuda:{i} matmul OK checksum={s:.1f}")
json.dump({"device_count":2,"devices":names,"ok":True},open("/kaggle/working/device_check.json","w"),indent=2)

# cell 1: obtain loopkit source (Kaggle github_sources is unreliable via CLI)
cands=[d for d in glob.glob("/kaggle/input/*") if os.path.isdir(d)]
repo=next((d for d in cands if (Path(d)/"pyproject.toml").exists() and "loopkit" in os.path.basename(d).lower()),None)
if repo is None:
    target="/kaggle/working/loopkit"
    if not (Path(target)/"pyproject.toml").exists():
        subprocess.run(["git","clone","-q","https://github.com/10xdev4u-alt/loopkit.git",target],check=True)
    repo=target
sys.path.insert(0,repo)
bad=[f for f in glob.glob(repo+"/**/*",recursive=True) if f.endswith((".pt",".pth",".ckpt",".safetensors",".bin"))]
assert not bad, f"weights in repo source: {bad[:5]}"
print("repo ready:",repo)

# cell 2: run on both GPUs
import numpy as np, time
from loopkit.nvfp4 import quantize_nvfp4, ans_encode, ans_decode, dequantize_nvfp4
from loopkit.geometry import spectral_break_even
rng=np.random.default_rng(20261123)
rows=[]
for gid in range(2):
    dev=torch.device(f"cuda:{gid}")
    t=time.perf_counter()
    x=rng.standard_normal((1024,1024)).astype("float32")
    tq=quantize_nvfp4(x)
    u=ans_decode(ans_encode(tq))
    assert np.array_equal(tq.codes,u.codes) and np.array_equal(tq.scales,u.scales)
    a=torch.from_numpy(dequantize_nvfp4(tq)).to(dev)
    y=(a@a.T).float().sum().item(); torch.cuda.synchronize(dev)
    enc=ans_encode(tq)
    rows.append({"gpu":gid,"device":str(dev),"checksum":round(y,2),
                 "bytes":int(x.nbytes),"ans_bytes":int(len(enc.payload)+len(enc.scale_payload)),
                 "seconds":round(time.perf_counter()-t,4)})
    print(rows[-1],flush=True)
gammas={"svd":spectral_break_even(0.54),"rot":spectral_break_even(0.393),"itq":spectral_break_even(0.30)}
json.dump({"config":{"seed":20261123,"shape":[1024,1024]},"per_gpu":rows,"gamma_star":gammas},
          open("/kaggle/working/summary.json","w"),indent=2)
print("KAGGLE PASS",gammas,flush=True)
